# Load Model

In [2]:
import json
import os
from pprint import pprint

import bitsandbytes as bnb
import pandas as pd
import torch
import torch.nn as nn
import transformers
from datasets import load_dataset
from trl import DPOConfig, DPOTrainer

from peft import (
    LoraConfig,
    PeftConfig,
    PeftModel,
    get_peft_model,
    prepare_model_for_kbit_training,
)
from transformers import (
    AutoConfig,
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)

MODEL_NAME = "Qwen/Qwen2.5-32B-Instruct"
# MODEL_NAME = "unsloth/Llama-3.2-1B" # Try Llama if you want

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)


model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    trust_remote_code=True,
    quantization_config=bnb_config,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/17 [00:00<?, ?it/s]

In [3]:
# Prepare input
text = "Salut je m'appelle"
inputs = tokenizer(text, return_tensors="pt").to("cuda")  # Ensure tensors are moved to GPU

# Generate text
outputs = model.generate(**inputs, max_new_tokens=20)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Salut je m'appelle Camille et j'ai 19 ans, j'habite à Paris et je suis en


In [5]:
import pandas as pd
articles_df = pd.read_csv("/users/eleves-b/2022/wajdi.maatouk/NLP_project/french_wikipedia_section_filtered_articles.csv")
articles_df.head()

,Category,Title,Summary,Text
0,Littérature,Hans Robert Jauss,"Hans Robert Jauss, né le 12 décembre 1921 à Gö...",Jauss s'est principalement fait connaitre dans...
1,Économie,Industrie du sexe,L'industrie du sexe recouvre l'ensemble du com...,"Certaines personnes, hommes ou femmes peuvent ..."
2,Littérature,Walter Muschg,"Walter Muschg, né le 21 mai 1898 à Witikon (pr...",Walter Muschg vient au monde en 1898 à Witikon...
3,Littérature,Joseph et ses frères,Joseph et ses frères (en allemand Joseph und s...,La tétralogie de Joseph est indissociable de l...
4,Littérature,Les Deux Frères (conte de Grimm),Les Deux Frères est le titre habituel d'un con...,"Dans la première édition du recueil (1812), le..."


In [11]:
random_article = articles_df.sample(1)
random_article

,Category,Title,Summary,Text
4898,Géographie,Climat du Sahara,"Le climat du Sahara, ou plutôt les climats du ...",## Types de climats sahariens\nOn compte génér...


In [16]:
import re

article_text  = random_article['Text'].values[0]

print(article_text)

## Types de climats sahariens
On compte généralement cinq types de zones climatiques qui caractérisent les différents climats du Sahara.

## Précipitations
Dans sa globalité, le Sahara est très aride et très sec. Sa zone centrale hyper-aride constitue une des régions les plus arides et plus sèches au monde. Le seul désert connaissant des précipitations annuelles similaires à celles de cette grande zone est le désert d'Atacama et encore, sur une étendue infiniment moindre comparée à celle du Sahara. Les précipitations moyennes annuelles varient en fonction des différentes zones climatiques du désert. Plus encore que leur très faible quantité, c'est également leur variabilité et leur irrégularité inter-mensuelle et interannuelle qui est exceptionnelle, surtout dans la zone saharo-sahélienne où il est normal qu'il ne pleuve point pendant 9 ou 10 mois consécutifs mais qu'il tombe près de la totalité des précipitations annuelles moyennes en quelques jours seulement, pendant les mois les plu

In [13]:
def generate_summary(article_text, model, tokenizer, max_new_tokens=200):
    """
    Generate a summary using the provided model and tokenizer.
    
    Parameters:
    - article_text (str): The input article text.
    - model: The pre-trained language model.
    - tokenizer: The tokenizer corresponding to the model.
    - max_new_tokens (int): Maximum number of new tokens in the summary.
    
    Returns:
    - str: The generated summary.
    """
    # Define summarization prompt in French
    prompt = "Résumez l'article suivant de manière concise :\n" + article_text
    
    # Tokenize input text with prompt
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        padding="longest"
    ).to(model.device)
    
    # Generate summary
    summary_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        length_penalty=2.0,
        num_beams=5,
        early_stopping=True
    )
    
    # Decode and return summary
    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

# Example usage:
summary = generate_summary(article_text, model, tokenizer)



In [15]:
print("Résumé généré :")
prompt = "Résumez l'article suivant de manière concise :\n" + article_text
print(summary[len(prompt):])

Résumé généré :
 La variabilité interannuelle des pluies est également très importante dans la zone saharo-sahélienne : par exemple, sur les 182 mm de précipitations annuelles qui tombent en moyenne à Tombouctou au Mali, la quantité de précipitations annuelles peut varier de moins de 100 mm à plus de 300 mm d'une année à l'autre; sur les 162 mm de précipitations annuelles qui tombent en moyenne à Khartoum au Soudan, la quantité de précipitations annuelles peut varier de moins de 100 mm à plus de 300 mm d'une année à l'autre; sur les 73 mm de précipitations annuelles qui tombent en moyenne à Atar en Mauritanie, la quantité de précipitations annuelles peut varier de moins de 50


In [1]:
import pandas as pd
articles = pd.read_csv("/users/eleves-b/2022/wajdi.maatouk/NLP_project/french_wikipedia_filtered_articles.csv")
articles.head()

,Category,Title,Summary,Text
0,Culture,Néférourê,Néférourê (La Beauté de Rê) est la fille aînée...,Néférourê (La Beauté de Rê) est la fille aînée...
1,Économie,Industrie du sexe,L'industrie du sexe recouvre l'ensemble du com...,L'industrie du sexe recouvre l'ensemble du com...
2,Littérature,Hans Robert Jauss,"Hans Robert Jauss, né le 12 décembre 1921 à Gö...","Hans Robert Jauss, né le 12 décembre 1921 à Gö..."
3,Sport,Saber Desfarges,"Saber Desfarges, né en 1989 à Clermont-Ferrand...","Saber Desfarges, né en 1989 à Clermont-Ferrand..."
4,Littérature,Walter Muschg,"Walter Muschg, né le 21 mai 1898 à Witikon (pr...","Walter Muschg, né le 21 mai 1898 à Witikon (pr..."


In [3]:
random_article = articles.sample(1)
print(random_article['Title'].values[0])
print(random_article['Text'].values[0])

Asiento
Un asiento est un type de convention qui avait cours dans l'ancienne monarchie espagnole. Il conférait à des acteurs privés le monopole d'exercer une compétence de l’État : commerce des esclaves noirs en provenance d'Afrique, prélèvement d'un impôt, transfert de fonds, exploitation d'une route commerciale (notamment celles vers les colonies espagnoles), etc. Ce monopole était concédé pour un temps limité, moyennant le versement d'une redevance à la couronne. Les asientos pouvaient être concédés à des individus, des banques, des entreprises, voire des États étrangers et concernaient tous les aspects de la vie économique du pays. Ils constituaient un moyen courant de gager les emprunts ou payer les dettes de la monarchie, pouvaient se revendre, se partager ou sous-traiter. Ils ressemblent ainsi un peu à la pratique de l'affermage, dans le royaume de France sous l'Ancien Régime.

L'asiento des esclaves noirs
Il existe donc des asientos pour tous types de produits coloniaux, mais l